# 📊 Análisis Exploratorio de Datos (EDA) Extensivo - Customer Support

Este notebook presenta un Análisis Exploratorio de Datos (EDA) sumamente profundo, estructurado en **18 puntos de análisis**, para el dataset de soporte al cliente. 

Nuestro objetivo es desentrañar patrones de comportamiento, temporalidad, eficiencia operativa y experiencia del cliente para generar insights de negocio accionables.

## 1. Importación de Librerías y Configuración
Cargamos las librerías necesarias para la manipulación de datos, cálculos estadísticos y generación de visualizaciones de alto impacto.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Configuraciones de visualización
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="mako")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["font.size"] = 11

## 2. Carga del Dataset
Cargamos la muestra estratificada del conjunto de datos original. Esta muestra fue seleccionada conservando las proporciones reales de industrias y resultados.

In [ ]:
data_path = "../data/customer_support_sample.csv"
df = pd.read_csv(data_path)
print(f"Dataset cargado en memoria exitosamente.")

## 3. Inspección Básica del Conjunto de Datos
Miramos las dimensiones generales y las primeras/últimas filas para entender la estructura a simple vista.

In [ ]:
print(f"Dimensiones (Filas, Columnas): {df.shape}")
display(df.head(3))
display(df.tail(2))

## 4. Estructura y Tipos de Datos
Analizamos cómo Pandas ha interpretado cada variable (strings, enteros, fechas).

In [ ]:
df.info()

## 5. Calidad de Datos (Data Quality): Valores Nulos
Evaluamos si existen campos vacíos que puedan sesgar nuestro análisis o requerir imputación.

In [ ]:
missing_data = df.isnull().sum()
missing_data = missing_data[missing_data > 0]
if missing_data.empty:
    print("✅ No se encontraron valores nulos en el dataset.")
else:
    print("⚠️ Variables con valores nulos:")
    print(missing_data)

## 6. Calidad de Datos: Duplicados y Cardinalidad
Identificamos la cantidad de valores únicos por columna y confirmamos que no existan registros 100% duplicados.

In [ ]:
print(f"Registros completamente duplicados: {df.duplicated().sum()}")

cardinality = df.nunique().to_frame(name="Valores Únicos").sort_values(by="Valores Únicos", ascending=False)
display(cardinality)

## 7. Ingeniería de Datos Básica (Limpieza y Formateo)
Aseguramos que la columna de tiempo tenga formato `datetime` y extraemos componentes útiles (día, hora, mes).

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.day_name()
df["month"] = df["timestamp"].dt.month_name()

df = df.dropna(subset=["timestamp"]) # Eliminar filas con fechas malformadas si las hubiera
print("Variables temporales creadas: hour, day_of_week, month.")

## 8. Análisis Univariado: Variables Categóricas Principales
Visualizamos la distribución macro de la data: Industria, Canal y Lenguaje.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

sns.countplot(data=df, y="industry", order=df["industry"].value_counts().index, ax=axes[0], palette="viridis")
axes[0].set_title("Distribución por Industria")

sns.countplot(data=df, x="channel", order=df["channel"].value_counts().index, ax=axes[1], palette="crest")
axes[1].set_title("Uso de Canales de Soporte")

sns.countplot(data=df, x="language", order=df["language"].value_counts().index, ax=axes[2], palette="magma")
axes[2].set_title("Idiomas Atendidos")

plt.tight_layout()
plt.show()

## 9. Análisis Univariado: Sentimiento y Urgencia del Cliente
Dos métricas vitales para el CSAT (Customer Satisfaction).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors_sentiment = {"negative": "#e74c3c", "neutral": "#95a5a6", "positive": "#2ecc71"}
sns.countplot(data=df, x="overall_sentiment", order=["negative", "neutral", "positive"], palette=colors_sentiment, ax=axes[0])
axes[0].set_title("Volumen por Sentimiento")

colors_urgency = {"critical": "#c0392b", "high": "#e67e22", "medium": "#f1c40f", "low": "#27ae60"}
sns.countplot(data=df, x="overall_urgency", order=["critical", "high", "medium", "low"], palette=colors_urgency, ax=axes[1])
axes[1].set_title("Volumen por Nivel de Urgencia")

plt.tight_layout()
plt.show()

## 10. Análisis Temporal: Tendencias a lo Largo del Tiempo
Agrupamos por día y mes para entender cuándo se exigen más los sistemas de soporte.

In [ ]:
plt.figure(figsize=(15, 5))
days_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
sns.countplot(data=df, x="day_of_week", order=days_order, palette="ocean")
plt.title("Volumen de Contacto por Día de la Semana")
plt.xlabel("Día")
plt.ylabel("Cantidad de Tickets")
plt.show()

## 11. Análisis Temporal: Mapas de Calor de Horarios Punta (Heatmaps)
Un cruce entre el día de la semana y la hora para encontrar los "picos" de estrés operativo.

In [ ]:
pivot_time = pd.crosstab(df["day_of_week"], df["hour"])
pivot_time = pivot_time.reindex(days_order)

plt.figure(figsize=(16, 6))
sns.heatmap(pivot_time, cmap="YlOrRd", annot=False, linewidths=.5)
plt.title("Heatmap: Día de la Semana vs Hora del Día")
plt.xlabel("Hora (0-23)")
plt.ylabel("Día")
plt.show()

## 12. Análisis Bivariado: Industria vs Nivel de Urgencia
¿Hay industrias donde los clientes reportan problemas más urgentes?

In [ ]:
industry_urgency = pd.crosstab(df["industry"], df["overall_urgency"], normalize="index") * 100
industry_urgency = industry_urgency[["critical", "high", "medium", "low"]]

industry_urgency.plot(kind="barh", stacked=True, figsize=(14, 7), color=[colors_urgency[k] for k in ["critical", "high", "medium", "low"]])
plt.title("Proporción de Urgencia por Industria")
plt.xlabel("Porcentaje (%)")
plt.legend(title="Urgencia", bbox_to_anchor=(1.05, 1))
plt.show()

## 13. Análisis Bivariado: Sentimiento según el Canal de Soporte
¿Un canal específico genera mayor frustración (ej. tiempos largos en chat)?

In [ ]:
channel_sentiment = pd.crosstab(df["channel"], df["overall_sentiment"], normalize="index") * 100
channel_sentiment = channel_sentiment[["negative", "neutral", "positive"]]

channel_sentiment.plot(kind="bar", stacked=True, figsize=(10, 6), color=[colors_sentiment[k] for k in ["negative", "neutral", "positive"]])
plt.title("Sentimiento del Cliente por Canal de Comunicación")
plt.ylabel("Porcentaje (%)")
plt.xticks(rotation=0)
plt.legend(title="Sentimiento", loc="upper right")
plt.show()

## 14. Análisis de Resultados (Outcomes)
Analizamos la capacidad resolutiva: qué porcentaje se resuelve y cuánto se escala.

In [ ]:
plt.figure(figsize=(10, 5))
outcomes = df["outcome"].value_counts(normalize=True) * 100
sns.barplot(x=outcomes.values, y=outcomes.index, palette="flare")
plt.title("Distribución Porcentual de Outcomes")
plt.xlabel("Porcentaje (%)")
for index, value in enumerate(outcomes.values):
    plt.text(value, index, f"{value:.1f}%", va="center")
plt.show()

## 15. Profundidad de Conversación (Turnos)
¿Cuántas idas y vueltas requiere un problema en promedio? (Turn_index)

In [ ]:
plt.figure(figsize=(12, 5))
sns.histplot(df["turn_index"], bins=30, kde=True, color="teal")
plt.title("Distribución de Turnos por Interacción")
plt.xlabel("Índice de Turno (Profundidad de la charla)")
plt.show()

## 16. Análisis de Intenciones (Top Intents)
Cuáles son las razones más recurrentes del contacto por parte de los clientes.

In [ ]:
top_intents = df["primary_intent"].value_counts().head(10)
plt.figure(figsize=(12, 6))
sns.barplot(x=top_intents.values, y=top_intents.index, palette="cubehelix")
plt.title("Top 10 Intenciones Principales del Cliente")
plt.xlabel("Cantidad de Contactos")
plt.show()

## 17. Feature Engineering Básico: Longitud del Texto
¿Escriben más los clientes cuando están frustrados o su problema es crítico? Creamos una métrica de longitud de texto.

In [ ]:
df["text_length"] = df["text"].astype(str).apply(len)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.boxplot(data=df[df["role"]=="customer"], x="overall_sentiment", y="text_length", ax=axes[0], palette=colors_sentiment)
axes[0].set_title("Longitud del Mensaje del Cliente vs Sentimiento")

sns.boxplot(data=df[df["role"]=="customer"], x="overall_urgency", y="text_length", order=["critical", "high", "medium", "low"], ax=axes[1], palette=colors_urgency)
axes[1].set_title("Longitud del Mensaje del Cliente vs Urgencia")
plt.tight_layout()
plt.show()

## 18. Conclusiones y Siguientes Pasos

**Hallazgos Clave:**
1. **Temporalidad**: Existen claros patrones horarios donde los canales deben ser reforzados con más agentes.
2. **Canales y Sentimiento**: Ciertos canales parecen acumular más sentimiento negativo, lo que podría indicar fricciones de UX (User Experience) o tiempos de espera (SLA) muy altos.
3. **Urgencia y Esfuerzo**: Los clientes que manifiestan problemas *críticos* tienden a enviar mensajes considerablemente más largos, lo que sugiere que expresan mayor detalle o frustración.

**Siguientes Pasos:**
- Entrenar un modelo de Machine Learning (NLP) para clasificar automáticamente el `primary_intent`.
- Desplegar el modelo en el enrutador de tickets para derivar urgencias críticas directamente a especialistas.
- Utilizar este notebook de base para alimentar dashboards dinámicos (Power BI / Tableau).